# Proyecto: Valorización energética de biomasa de palma - Magdalena
## Modelo Python/Pyomo basado en Capítulo 5 de tesis doctoral

## 0. Imports generales
Ejecutar siempre esta celda primero después de reiniciar el kernel

In [1]:
import pandas as pd

df = pd.read_excel('Escenarios.xlsx', sheet_name='Datos_Base')
df.head()

,Parametros,Unidad,Planta extractora,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23
0,NaN,NaN,Planta A,Planta B,Planta C,Planta D,Planta E,Planta F,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Capacidad de procesamiento,tRFF∙h-1,30,40,30,24,24,20,NaN,NaN,...,Diésel,NaN,NaN,NaN,Planta A,Planta B,Planta C,Planta D,Planta E,Planta F
2,Fruto procesado,tRFF∙año-1,153157,206207,177589,131909,75534,38620,NaN,NaN,...,0,904.804593,NaN,Biomasa,383.804934,0,0,0,0,0
3,Horas de operación,h∙año-1,5105,5155,5920,5496,3147,1931,NaN,NaN,...,5.336014,485.043673,NaN,Biogás,0,0,0,553,0,0
4,Tasa de extracción aceite,% ACP∙tRFF-1,20.71,20.55,19.68,19.9,20.77,19.3,NaN,NaN,...,56.122712,412.666996,NaN,Red nacional,95.902724,904.804593,356.544285,158,384.126622,455.842022


In [2]:
print(df.shape)
print(df['Parametros'].tolist())


(71, 24)
[nan, 'Capacidad de procesamiento', 'Fruto procesado ', 'Horas de operación', 'Tasa de extracción aceite', 'Aceite crudo producido', 'PKO producido', 'Humedad PKO', 'Esterilización', 'Vapor de esterilización', nan, 'Deshidratación del fruto ', 'Condensados de esterilización', nan, 'Vapor evaporador', nan, 'Eficiencia energpetica', nan, 'Desfrutado', 'Raquis vacío ', 'Humedad raquis', 'Masa que pasa por digestor', 'Humedad MPD', 'Digestión y prensado', 'Vapor de digestión', 'Licor de prensa', 'Aceite crudo en LP', 'Agua en LP', 'Residuos orgánicos en LP', 'Fibra de mesocarpio', 'Humedad de mesocarpio', 'Nueces', 'Humedad de nueces', 'Clarificación', 'Agua de dilución', 'Temperatura LP', 'Temperatura aceite clarificado', 'Aceite clarificado', 'Vapor de clarificacion', 'Lodos de clarificación', 'Recuperación de aceite', 'Agua de recuperación', 'Aceite recuperado', 'Efluente', 'Almacenamiento', 'Vapor de almacenamiento', 'Temperatura despacho', 'Secado de nueces', 'Vapor de secado

In [3]:
import pandas as pd

# Leer solo las columnas A-H (Parametros, Unidad, y las 6 plantas)
df_raw = pd.read_excel('Escenarios.xlsx', sheet_name='Datos_Base', header=None, usecols='A:H')

# La fila con índice 1 (segunda fila del Excel) tiene los nombres de las plantas
plantas = df_raw.iloc[1, 2:8].tolist()
print("Plantas detectadas:", plantas)

# Los datos reales empiezan en la fila con índice 2
df_datos = df_raw.iloc[2:].copy()
df_datos.columns = ['Parametro', 'Unidad'] + plantas
df_datos = df_datos.reset_index(drop=True)

# Formato "tidy": una fila por combinación planta-parámetro
df_tidy = df_datos.melt(id_vars=['Parametro', 'Unidad'], var_name='Planta', value_name='Valor')
df_tidy = df_tidy.dropna(subset=['Valor'])

print(df_tidy.shape)
df_tidy.head(15)

Plantas detectadas: ['Planta A', 'Planta B', 'Planta C', 'Planta D', 'Planta E', 'Planta F']
(313, 4)


,Parametro,Unidad,Planta,Valor
0,Capacidad de procesamiento,tRFF∙h-1,Planta A,30
1,Fruto procesado,tRFF∙año-1,Planta A,153157
2,Horas de operación,h∙año-1,Planta A,5105
3,Tasa de extracción aceite,% ACP∙tRFF-1,Planta A,20.71
4,Aceite crudo producido,tACP∙año-1,Planta A,31719
5,PKO producido,tPKO∙año-1,Planta A,7636
6,Humedad PKO,%,Planta A,6.38
8,Vapor de esterilización,kgvapor∙h-1,Planta A,6371
9,NaN,NaN,Planta A,212.366667
10,Deshidratación del fruto,%,Planta A,11.8


In [4]:
import pandas as pd

df_raw = pd.read_excel('Escenarios.xlsx', sheet_name='Datos_Base', header=None, usecols='A:H')
plantas = df_raw.iloc[1, 2:8].tolist()

df_datos = df_raw.iloc[2:].copy()
df_datos.columns = ['Parametro', 'Unidad'] + plantas
df_datos = df_datos.reset_index(drop=True)

# Identificar filas de encabezado de etapa: tienen nombre de Parametro
# pero ningún valor en ninguna planta (ej. "Esterilización", "Desfrutado")
is_header = df_datos['Parametro'].notna() & df_datos[plantas].isna().all(axis=1)

# Crear columna Etapa, propagando el nombre hacia abajo desde cada encabezado
df_datos['Etapa'] = df_datos['Parametro'].where(is_header).ffill()

# Quitar las filas de encabezado (ya cumplieron su función de etiquetar la Etapa)
df_datos = df_datos[~is_header].reset_index(drop=True)

# Rellenar Parametro para las filas "huérfanas" (valor específico por tRFF procesado)
df_datos['Parametro'] = df_datos['Parametro'].ffill()
dup_mask = df_datos.duplicated(subset=['Etapa', 'Parametro'], keep='first')
df_datos.loc[dup_mask, 'Parametro'] = df_datos.loc[dup_mask, 'Parametro'] + ' (específico por tRFF)'
df_datos.loc[dup_mask, 'Unidad'] = df_datos.loc[dup_mask, 'Unidad'].fillna('por tRFF procesado')

# Formato tidy final
df_tidy = df_datos.melt(id_vars=['Parametro', 'Unidad', 'Etapa'], var_name='Planta', value_name='Valor')
df_tidy = df_tidy.dropna(subset=['Valor']).reset_index(drop=True)

print(df_tidy.shape)
print(df_tidy['Etapa'].unique())
df_tidy.head(20)

(313, 5)
[nan 'Esterilización' 'Desfrutado' 'Digestión y prensado' 'Clarificación'
 'Recuperación de aceite' 'Almacenamiento' 'Secado de nueces'
 'Flujo de aire de secado' 'Rotura de nuez' 'Calentamiento agua proceso'
 'Caldera']


,Parametro,Unidad,Etapa,Planta,Valor
0,Capacidad de procesamiento,tRFF∙h-1,NaN,Planta A,30
1,Fruto procesado,tRFF∙año-1,NaN,Planta A,153157
2,Horas de operación,h∙año-1,NaN,Planta A,5105
3,Tasa de extracción aceite,% ACP∙tRFF-1,NaN,Planta A,20.71
4,Aceite crudo producido,tACP∙año-1,NaN,Planta A,31719
5,PKO producido,tPKO∙año-1,NaN,Planta A,7636
6,Humedad PKO,%,NaN,Planta A,6.38
7,Vapor de esterilización,kgvapor∙h-1,Esterilización,Planta A,6371
8,Vapor de esterilización (específico por tRFF),por tRFF procesado,Esterilización,Planta A,212.366667
9,Deshidratación del fruto,%,Esterilización,Planta A,11.8


In [5]:
import pandas as pd

df_raw = pd.read_excel('Escenarios.xlsx', sheet_name='Datos_Base', header=None, usecols='A:H')
plantas = df_raw.iloc[1, 2:8].tolist()

df_datos = df_raw.iloc[2:].copy()
df_datos.columns = ['Parametro', 'Unidad'] + plantas
df_datos = df_datos.reset_index(drop=True)

# Identificar filas de encabezado de etapa: tienen nombre de Parametro
# pero ningún valor en ninguna planta (ej. "Esterilización", "Desfrutado")
is_header = df_datos['Parametro'].notna() & df_datos[plantas].isna().all(axis=1)

# Crear columna Etapa, propagando el nombre hacia abajo desde cada encabezado
df_datos['Etapa'] = df_datos['Parametro'].where(is_header).ffill()

# Quitar las filas de encabezado (ya cumplieron su función de etiquetar la Etapa)
df_datos = df_datos[~is_header].reset_index(drop=True)

# Rellenar Parametro para las filas "huérfanas" (valor específico por tRFF procesado)
df_datos['Parametro'] = df_datos['Parametro'].ffill()
dup_mask = df_datos.duplicated(subset=['Etapa', 'Parametro'], keep='first')
df_datos.loc[dup_mask, 'Parametro'] = df_datos.loc[dup_mask, 'Parametro'] + ' (específico por tRFF)'
df_datos.loc[dup_mask, 'Unidad'] = df_datos.loc[dup_mask, 'Unidad'].fillna('por tRFF procesado')

# Formato tidy final
df_tidy = df_datos.melt(id_vars=['Parametro', 'Unidad', 'Etapa'], var_name='Planta', value_name='Valor')
df_tidy = df_tidy.dropna(subset=['Valor']).reset_index(drop=True)
df_tidy['Etapa'] = df_tidy['Etapa'].fillna('General')

print(df_tidy.shape)
print(df_tidy['Etapa'].unique())
df_tidy.head(20)

(313, 5)
['General' 'Esterilización' 'Desfrutado' 'Digestión y prensado'
 'Clarificación' 'Recuperación de aceite' 'Almacenamiento'
 'Secado de nueces' 'Flujo de aire de secado' 'Rotura de nuez'
 'Calentamiento agua proceso' 'Caldera']


,Parametro,Unidad,Etapa,Planta,Valor
0,Capacidad de procesamiento,tRFF∙h-1,General,Planta A,30
1,Fruto procesado,tRFF∙año-1,General,Planta A,153157
2,Horas de operación,h∙año-1,General,Planta A,5105
3,Tasa de extracción aceite,% ACP∙tRFF-1,General,Planta A,20.71
4,Aceite crudo producido,tACP∙año-1,General,Planta A,31719
5,PKO producido,tPKO∙año-1,General,Planta A,7636
6,Humedad PKO,%,General,Planta A,6.38
7,Vapor de esterilización,kgvapor∙h-1,Esterilización,Planta A,6371
8,Vapor de esterilización (específico por tRFF),por tRFF procesado,Esterilización,Planta A,212.366667
9,Deshidratación del fruto,%,Esterilización,Planta A,11.8


In [6]:
import pandas as pd

df_raw = pd.read_excel('Escenarios.xlsx', sheet_name='Datos_Base', header=None, usecols='A:H')
plantas = df_raw.iloc[1, 2:8].tolist()

df_datos = df_raw.iloc[2:].copy()
df_datos.columns = ['Parametro', 'Unidad'] + plantas
df_datos = df_datos.reset_index(drop=True)

# Identificar filas de encabezado de etapa: tienen nombre de Parametro
# pero ningún valor en ninguna planta (ej. "Esterilización", "Desfrutado")
is_header = df_datos['Parametro'].notna() & df_datos[plantas].isna().all(axis=1)

# Crear columna Etapa, propagando el nombre hacia abajo desde cada encabezado
df_datos['Etapa'] = df_datos['Parametro'].where(is_header).ffill()

# Quitar las filas de encabezado (ya cumplieron su función de etiquetar la Etapa)
df_datos = df_datos[~is_header].reset_index(drop=True)

# Rellenar Parametro para las filas "huérfanas" (valor específico por tRFF procesado)
df_datos['Parametro'] = df_datos['Parametro'].ffill()
dup_mask = df_datos.duplicated(subset=['Etapa', 'Parametro'], keep='first')
df_datos.loc[dup_mask, 'Parametro'] = df_datos.loc[dup_mask, 'Parametro'] + ' (específico por tRFF)'
df_datos.loc[dup_mask, 'Unidad'] = df_datos.loc[dup_mask, 'Unidad'].fillna('por tRFF procesado')

# Formato tidy final
df_tidy = df_datos.melt(id_vars=['Parametro', 'Unidad', 'Etapa'], var_name='Planta', value_name='Valor')
df_tidy = df_tidy.dropna(subset=['Valor']).reset_index(drop=True)
df_tidy['Etapa'] = df_tidy['Etapa'].fillna('General')

print(df_tidy.shape)
print(df_tidy['Etapa'].unique())
df_tidy.head(20)

df_tidy['Etapa'] = df_tidy['Etapa'].fillna('General')

# Eliminar las filas de "Eficiencia" - se recalcularán desde cero con el modelo Pyomo
# a partir del balance de masa y energía (no de fórmulas de Excel potencialmente rotas)
es_eficiencia = df_tidy['Parametro'].str.contains('Eficiencia', case=False, na=False)
df_datos_base = df_tidy[~es_eficiencia].reset_index(drop=True)

print(df_datos_base.shape)
print(df_datos_base['Etapa'].unique())
df_datos_base.tail(20)

(313, 5)
['General' 'Esterilización' 'Desfrutado' 'Digestión y prensado'
 'Clarificación' 'Recuperación de aceite' 'Almacenamiento'
 'Secado de nueces' 'Flujo de aire de secado' 'Rotura de nuez'
 'Calentamiento agua proceso' 'Caldera']
(307, 5)
['General' 'Esterilización' 'Desfrutado' 'Digestión y prensado'
 'Clarificación' 'Recuperación de aceite' 'Almacenamiento'
 'Secado de nueces' 'Flujo de aire de secado' 'Rotura de nuez'
 'Calentamiento agua proceso' 'Caldera']


,Parametro,Unidad,Etapa,Planta,Valor
287,Lodos de clarificación,kgLodos∙h-1,Clarificación,Planta F,9271
288,Agua de recuperación,kgagua∙h-1,Recuperación de aceite,Planta F,2605
289,Aceite recuperado,kgaceite∙h-1,Recuperación de aceite,Planta F,455
290,Efluente,kgefluente∙h-1,Recuperación de aceite,Planta F,11422
291,Vapor de almacenamiento,kgvapor∙h-1,Almacenamiento,Planta F,932
292,Temperatura despacho,°C,Almacenamiento,Planta F,55
293,Vapor de secado,kgvapor∙h-1,Secado de nueces,Planta F,992
294,Temperatura aire de secado,°C,Flujo de aire de secado,Planta F,60
295,Humedad de secado,°C,Flujo de aire de secado,Planta F,7.4
296,Cuesco,tCuesco∙año-1,Rotura de nuez,Planta F,2703


In [7]:
print(df_datos_base[['Parametro', 'Unidad', 'Etapa']].drop_duplicates().to_string())

                                              Parametro              Unidad                       Etapa
0                            Capacidad de procesamiento            tRFF∙h-1                     General
1                                      Fruto procesado           tRFF∙año-1                     General
2                                    Horas de operación             h∙año-1                     General
3                             Tasa de extracción aceite        % ACP∙tRFF-1                     General
4                                Aceite crudo producido          tACP∙año-1                     General
5                                         PKO producido          tPKO∙año-1                     General
6                                           Humedad PKO                   %                     General
7                               Vapor de esterilización         kgvapor∙h-1              Esterilización
8         Vapor de esterilización (específico por tRFF)  por tRF

In [8]:
pd.set_option('display.max_rows', None)
df_datos_base[['Parametro', 'Unidad', 'Etapa']].drop_duplicates()

,Parametro,Unidad,Etapa
0,Capacidad de procesamiento,tRFF∙h-1,General
1,Fruto procesado,tRFF∙año-1,General
2,Horas de operación,h∙año-1,General
3,Tasa de extracción aceite,% ACP∙tRFF-1,General
4,Aceite crudo producido,tACP∙año-1,General
5,PKO producido,tPKO∙año-1,General
6,Humedad PKO,%,General
7,Vapor de esterilización,kgvapor∙h-1,Esterilización
8,Vapor de esterilización (específico por tRFF),por tRFF procesado,Esterilización
9,Deshidratación del fruto,%,Esterilización


In [9]:
df_datos_base[['Parametro', 'Unidad', 'Etapa']].drop_duplicates().to_csv('parametros_verificacion.csv', index=False)

In [10]:
df_datos_base.loc[df_datos_base['Parametro'] == 'Humedad de secado', 'Unidad'] = '%'


In [11]:
import numpy as np

fila_pendiente = pd.DataFrame({
    'Parametro': ['Flujo de aire de secado'] * len(plantas),
    'Unidad': ['kgaire seco∙h-1'] * len(plantas),
    'Etapa': ['Secado de nueces'] * len(plantas),
    'Planta': plantas,
    'Valor': [np.nan] * len(plantas)
})

df_datos_base = pd.concat([df_datos_base, fila_pendiente], ignore_index=True)

In [12]:
print(df_datos_base['Etapa'].unique())
df_datos_base[df_datos_base['Valor'].isna()]

['General' 'Esterilización' 'Desfrutado' 'Digestión y prensado'
 'Clarificación' 'Recuperación de aceite' 'Almacenamiento'
 'Secado de nueces' 'Flujo de aire de secado' 'Rotura de nuez'
 'Calentamiento agua proceso' 'Caldera']


,Parametro,Unidad,Etapa,Planta,Valor
307,Flujo de aire de secado,kgaire seco∙h-1,Secado de nueces,Planta A,NaN
308,Flujo de aire de secado,kgaire seco∙h-1,Secado de nueces,Planta B,NaN
309,Flujo de aire de secado,kgaire seco∙h-1,Secado de nueces,Planta C,NaN
310,Flujo de aire de secado,kgaire seco∙h-1,Secado de nueces,Planta D,NaN
311,Flujo de aire de secado,kgaire seco∙h-1,Secado de nueces,Planta E,NaN
312,Flujo de aire de secado,kgaire seco∙h-1,Secado de nueces,Planta F,NaN


In [13]:
df_datos_base.to_csv('datos_base_limpio.csv', index=False)

In [14]:
raw = pd.read_excel('Escenarios.xlsx', sheet_name='Biomass', header=None)

# Materiales: fila índice 4 (excel fila 5), columnas C,E,G,I,K (cada 2 columnas)
materiales = raw.iloc[4, 2:12:2].tolist()
print("Materiales detectados:", materiales)

# Propiedades y unidades: columnas A y B, filas 8 a 15 (índices 7 a 14)
propiedades = raw.iloc[7:15, 0].tolist()
unidades = raw.iloc[7:15, 1].ffill().tolist()  # rellenar unidad hacia abajo (C,H,O,N,S comparten "% dry basis")

registros = []
for i, mat in enumerate(materiales):
    col_avg = 2 + i*2
    col_std = 3 + i*2
    for j, fila_excel in enumerate(range(7, 15)):
        registros.append({
            'Material': mat,
            'Propiedad': propiedades[j],
            'Unidad': unidades[j],
            'Promedio': raw.iloc[fila_excel, col_avg],
            'Desviacion_std': raw.iloc[fila_excel, col_std]
        })

df_biomasa = pd.DataFrame(registros)
df_biomasa

Materiales detectados: ['EFB', 'Fiber', 'PKS', 'Pressed EFB', 'Dry EFB']


,Material,Propiedad,Unidad,Promedio,Desviacion_std
0,EFB,Moisture,(% wet basis),0.650000,5.1
1,EFB,C,(% dry basis),0.440600,2.5
2,EFB,H,(% dry basis),0.060800,0.7
3,EFB,O,(% dry basis),0.358500,4.1
4,EFB,N,(% dry basis),0.005800,0.7
5,EFB,S,(% dry basis),0.001200,0.4
6,EFB,HHV,(kJ/kg),18363.408000,2.4
7,EFB,LHV,(kJ/kg),6425.139466,NaN
8,Fiber,Moisture,(% wet basis),0.350000,2.4
9,Fiber,C,(% dry basis),0.453100,3.2


In [15]:
import pandas as pd

# ==========================================================
# PASO 1: Limpiar la tabla de propiedades (filas 5-15)
# ==========================================================
raw = pd.read_excel('Escenarios.xlsx', sheet_name='Biomass', header=None)

materiales = raw.iloc[4, 2:12:2].tolist()
propiedades = raw.iloc[7:15, 0].tolist()
unidades = raw.iloc[7:15, 1].ffill().tolist()

registros = []
for i, mat in enumerate(materiales):
    col_avg = 2 + i*2
    col_std = 3 + i*2
    for j, fila_excel in enumerate(range(7, 15)):
        registros.append({
            'Material': mat,
            'Propiedad': propiedades[j],
            'Unidad': unidades[j],
            'Promedio': raw.iloc[fila_excel, col_avg],
            'Desviacion_std': raw.iloc[fila_excel, col_std]
        })

df_biomasa = pd.DataFrame(registros)
print("df_biomasa listo:", df_biomasa.shape)

# ==========================================================
# PASO 2: Tabla de capacidad calorífica (de la imagen)
# ==========================================================
df_calor_especifico = pd.DataFrame({
    'Material': ['Water', 'Nut shell', 'Kernel', 'Palm oil', 'Fiber', 'EFB', 'Mud, etc.'],
    'Peso_pct': [15, 6.8, 5.2, 23.5, 14.0, 22.0, 12],
    'Cp_kJ_kgK': [4.18, 1.88, 1.59, 1.46, 1.80, 1.67, 2.22]
})

# ==========================================================
# PASO 3: Función de correlación HHV "Present study" (Eq. 7)
# ==========================================================
def hhv_present_study(C, H, O, N, S):
    """C, H, O, N, S en % base seca (dry basis). Retorna HHV en MJ/kg."""
    return 0.3443*C + 1.192*H - 0.113*O - 0.024*N + 0.093*S

# ==========================================================
# PASO 4: Validación cruzada (usa df_biomasa del Paso 1)
# ==========================================================
df_wide = df_biomasa.pivot_table(index='Material', columns='Propiedad', values='Promedio').reset_index()

df_wide['HHV_calculado_MJkg'] = df_wide.apply(
    lambda row: hhv_present_study(row['C']*100, row['H']*100, row['O']*100, row['N']*100, row['S']*100),
    axis=1
)
df_wide['HHV_reportado_MJkg'] = df_wide['HHV'] / 1000

df_wide[['Material', 'HHV_calculado_MJkg', 'HHV_reportado_MJkg']]

df_biomasa listo: (40, 5)


Propiedad,Material,HHV_calculado_MJkg,HHV_reportado_MJkg
0,Dry EFB,18.363408,18.363408
1,EFB,18.363408,18.363408
2,Fiber,19.228173,19.228173
3,PKS,19.108257,19.108257
4,Pressed EFB,18.363408,18.363408


In [17]:
df_biomasa.to_csv('biomasa_propiedades.csv', index=False)
df_calor_especifico.to_csv('biomasa_calor_especifico.csv', index=False)

In [18]:
# ==========================================================
# PASO 4: Escalamiento de Pretratamiento térmico (regla de los 6/10)
# ==========================================================
CEPCI_2021 = 708.8
factor_cepci_2021_2026 = CEPCI_2026 / CEPCI_2021

COSTO_REF_2021 = 1_400_000                          # USD, año 2021
COSTO_REF_2026 = COSTO_REF_2021 * factor_cepci_2021_2026   # ajustado a USD 2026
CAPACIDAD_REF = 150_000                              # t/año
n = 0.6

def costo_pretratamiento_termico(capacidad_planta_t_ano):
    """Costo estimado de pretratamiento térmico (USD 2026) vía regla de escalamiento (n=0.6)."""
    return COSTO_REF_2026 * (capacidad_planta_t_ano / CAPACIDAD_REF) ** n

print(f"Costo de referencia ajustado a 2026: ${COSTO_REF_2026:,.0f} USD")

# Ejemplo: aplicarlo a las 6 plantas (usando 'Fruto procesado' de df_datos_base)
fruto_procesado = df_datos_base[df_datos_base['Parametro'] == 'Fruto procesado'].set_index('Planta')['Valor']
for planta, tRFF_ano in fruto_procesado.items():
    costo = costo_pretratamiento_termico(tRFF_ano)
    print(f"{planta}: {tRFF_ano:,.0f} tRFF/año -> ${costo:,.0f} USD (2026)")

NameError: name 'CEPCI_2026' is not defined

In [19]:
import pandas as pd
import numpy as np

# ==========================================================
# PASO 0: Recargar Datos_Base (por si el kernel se reinició)
# ==========================================================
raw = pd.read_excel('Escenarios.xlsx', sheet_name='Datos_Base', header=None, usecols='A:H')
plantas = raw.iloc[1, 2:8].tolist()
df_datos = raw.iloc[2:].copy()
df_datos.columns = ['Parametro', 'Unidad'] + plantas
df_datos = df_datos.reset_index(drop=True)

is_header = df_datos['Parametro'].notna() & df_datos['Unidad'].isna() & df_datos[plantas].isna().all(axis=1)
df_datos['Etapa'] = df_datos['Parametro'].where(is_header).ffill()
df_datos = df_datos[~is_header].reset_index(drop=True)

df_datos['Parametro'] = df_datos['Parametro'].ffill()
dup_mask = df_datos.duplicated(subset=['Etapa', 'Parametro'], keep='first')
df_datos.loc[dup_mask, 'Parametro'] = df_datos.loc[dup_mask, 'Parametro'] + ' (específico por tRFF)'
df_datos.loc[dup_mask, 'Unidad'] = df_datos.loc[dup_mask, 'Unidad'].fillna('por tRFF procesado')

df_tidy = df_datos.melt(id_vars=['Parametro', 'Unidad', 'Etapa'], var_name='Planta', value_name='Valor')
df_tidy['Etapa'] = df_tidy['Etapa'].fillna('General')
df_tidy = df_tidy.dropna(subset=['Valor']).reset_index(drop=True)

df_tidy.loc[df_tidy['Parametro'] == 'Humedad de secado', 'Unidad'] = '%'

fila_pendiente = pd.DataFrame({
    'Parametro': ['Flujo de aire de secado'] * len(plantas),
    'Unidad': ['kgaire seco∙h-1'] * len(plantas),
    'Etapa': ['Secado de nueces'] * len(plantas),
    'Planta': plantas,
    'Valor': [np.nan] * len(plantas)
})

es_eficiencia = df_tidy['Parametro'].str.contains('Eficiencia', case=False, na=False)
df_datos_base = df_tidy[~es_eficiencia].reset_index(drop=True)
df_datos_base = pd.concat([df_datos_base, fila_pendiente], ignore_index=True)

print("df_datos_base listo:", df_datos_base.shape)

# ==========================================================
# PASO 1: Limpiar el resumen por área (CAPEX, filas 14-38 = índices 13-37)
# ==========================================================
raw_capex = pd.read_excel('Escenarios.xlsx', sheet_name='CAPEX', header=None)

registros_capex = []
for r in range(13, 38):
    area = raw_capex.iloc[r, 6]         # columna G
    total = raw_capex.iloc[r, 9]        # columna J
    especifico = raw_capex.iloc[r, 10]  # columna K
    unidad = raw_capex.iloc[r, 11]      # columna L
    if pd.notna(area):
        registros_capex.append({
            'Area': area,
            'Total_USD_ref': pd.to_numeric(total, errors='coerce'),
            'Valor_especifico': pd.to_numeric(especifico, errors='coerce'),
            'Unidad_especifico': unidad
        })

df_capex = pd.DataFrame(registros_capex)
print("df_capex:")
print(df_capex)

# ==========================================================
# PASO 2: Ajuste de todos los valores a USD 2026 (CEPCI)
# ==========================================================
CEPCI_2019 = 607.5
CEPCI_2021 = 708.8
CEPCI_2026 = 873.8  # estimado, ver nota metodológica

factor_cepci_2019 = CEPCI_2026 / CEPCI_2019
factor_cepci_2021 = CEPCI_2026 / CEPCI_2021

df_capex['Total_USD_2026'] = df_capex['Total_USD_ref'] * factor_cepci_2019
df_capex['Valor_especifico_2026'] = df_capex['Valor_especifico'] * factor_cepci_2019

print(df_capex[['Area', 'Total_USD_2026', 'Valor_especifico_2026']])

# ==========================================================
# PASO 3: Curva de costo de cogeneración (Malico et al., 2019)
# ==========================================================
def capex_cogeneracion_usd_kw(capacidad_kw):
    """CAPEX específico (USD/kW) según capacidad instalada. Fuente: Malico et al. (2019). Retorna en USD 2026."""
    valor_2019 = -900.6 * np.log(capacidad_kw) + 10841
    return valor_2019 * factor_cepci_2019

print("\nEjemplo curva cogeneración (1500 kW):", capex_cogeneracion_usd_kw(1500))

# ==========================================================
# PASO 4: Escalamiento de Pretratamiento térmico (regla de los 6/10)
# ==========================================================
COSTO_REF_2021 = 1_400_000
COSTO_REF_2026 = COSTO_REF_2021 * factor_cepci_2021
CAPACIDAD_REF = 150_000
n = 0.6

def costo_pretratamiento_termico(capacidad_planta_t_ano):
    return COSTO_REF_2026 * (capacidad_planta_t_ano / CAPACIDAD_REF) ** n

print(f"\nCosto de referencia ajustado a 2026: ${COSTO_REF_2026:,.0f} USD")

fruto_procesado = df_datos_base[df_datos_base['Parametro'] == 'Fruto procesado'].set_index('Planta')['Valor']
for planta, tRFF_ano in fruto_procesado.items():
    costo = costo_pretratamiento_termico(tRFF_ano)
    print(f"{planta}: {tRFF_ano:,.0f} tRFF/año -> ${costo:,.0f} USD (2026)")

df_datos_base listo: (313, 5)
df_capex:
                                         Area  Total_USD_ref  \
0                                   Recepción   1.827019e+05   
1                 Esterilización convencional   5.842191e+05   
2                     Esterilización continua   5.760070e+05   
3                          Prensado de raquis   1.117692e+05   
4                                 Desfrutado    1.332761e+05   
5           Extracción (digestión y prensado)   2.583898e+05   
6                      Clarificación dinamica   3.505156e+05   
7                      Clarificación estatica   2.804464e+05   
8                              Almacenamiento   2.014153e+05   
9                                 Palmisteria   5.534873e+05   
10                      Caldera media presión   1.141062e+06   
11                      Turbina Contrapresiòn   2.814681e+03   
12                    Redes e infraestructura   1.072970e+05   
13            Turbina Extracción condensación            NaN   


In [20]:
df_datos_base.to_csv('datos_base_limpio.csv', index=False)
df_biomasa.to_csv('biomasa_propiedades.csv', index=False)
df_calor_especifico.to_csv('biomasa_calor_especifico.csv', index=False)
df_capex.to_csv('capex_limpio_2026.csv', index=False)

In [21]:
import pandas as pd

# ==========================================================
# Tabla de factores de emisión (Gómez et al., 2017 - valores IPCC por defecto)
# ==========================================================
df_factores_emision = pd.DataFrame({
    'Gas': ['CH4', 'CH4', 'N2O', 'N2O'],
    'Fuente': ['Biomasa sólida', 'Biogás', 'Biomasa sólida', 'Biogás'],
    'Factor_kg_TJ': [30, 1, 4, 0.1],
    'GWP100_kgCO2eq_kg': [21, 21, 310, 310]
})

# Factores adicionales (validados/documentados)
FACTOR_CO2_DIESEL = 74100        # kg CO2/TJ (IPCC 2006, validado contra hoja original)
FACTOR_RED_NACIONAL = 0.1634     # kg CO2/kWh (163.4 gCO2/kWh - valor usado en publicación previa)
FACTOR_ABSORCION_PALMA = -135.04 # kg CO2/tRFF (crédito por absorción, cultivo de palma)

# ==========================================================
# Funciones de cálculo de emisiones
# ==========================================================
def emisiones_no_co2_biomasa(energia_TJ, fuente='Biomasa sólida'):
    """Emisiones de CH4 y N2O (no-CO2) por combustión de biomasa sólida o biogás, en kg CO2-eq.
    El CO2 de la combustión de biomasa/biogás se considera biogénico (neutro),
    por convención IPCC no se contabiliza en el inventario de GEI."""
    fila_ch4 = df_factores_emision[(df_factores_emision['Gas']=='CH4') & (df_factores_emision['Fuente']==fuente)].iloc[0]
    fila_n2o = df_factores_emision[(df_factores_emision['Gas']=='N2O') & (df_factores_emision['Fuente']==fuente)].iloc[0]
    ch4_eq = energia_TJ * fila_ch4['Factor_kg_TJ'] * fila_ch4['GWP100_kgCO2eq_kg']
    n2o_eq = energia_TJ * fila_n2o['Factor_kg_TJ'] * fila_n2o['GWP100_kgCO2eq_kg']
    return ch4_eq + n2o_eq  # kg CO2-eq

def emisiones_diesel(energia_TJ):
    """Emisiones de CO2 por combustión de diésel, en kg CO2."""
    return energia_TJ * FACTOR_CO2_DIESEL

def emisiones_red_nacional(electricidad_kWh):
    """Emisiones de CO2 por consumo de electricidad de la red nacional, en kg CO2."""
    return electricidad_kWh * FACTOR_RED_NACIONAL

def balance_neto_GEI(emisiones_proceso_kgCO2eq, tRFF_procesado):
    """Balance neto de GEI: emisiones del proceso menos crédito de absorción del cultivo de palma."""
    credito_absorcion = FACTOR_ABSORCION_PALMA * tRFF_procesado
    return emisiones_proceso_kgCO2eq + credito_absorcion

def excedente_electrico(electricidad_generada_kWh, electricidad_consumida_kWh):
    """Excedente eléctrico exportable (negativo si la planta no se autoabastece)."""
    return electricidad_generada_kWh - electricidad_consumida_kWh

# ==========================================================
# Validación contra Bloque 1 (línea base) - Planta A
# ==========================================================
tj_diesel_plantaA = 0.46090413989315915
tRFF_plantaA = 153157
print("Emisiones diésel Planta A:", emisiones_diesel(tj_diesel_plantaA), "≈ 34,152.99 (valor original de la hoja)")
print("Crédito absorción Planta A:", FACTOR_ABSORCION_PALMA * tRFF_plantaA, "kg CO2")

Emisiones diésel Planta A: 34152.996766083095 ≈ 34,152.99 (valor original de la hoja)
Crédito absorción Planta A: -20682321.279999997 kg CO2


In [22]:
df_factores_emision.to_csv('factores_emision.csv', index=False)

In [23]:
import pandas as pd

# ==========================================================
# Datos de demanda residencial departamental (Magdalena, 2025)
# ==========================================================
DEMANDA_RESIDENCIAL_RURAL_GWh = 30.3
DEMANDA_RESIDENCIAL_URBANA_GWh = 1209
DEMANDA_INDUSTRIAL_COMERCIO_GWh = 1190.7
DEMANDA_TOTAL_DEPARTAMENTAL_GWh = 2430  # validado: suma de las 3 anteriores

USUARIOS_RURALES = 87516
USUARIOS_URBANOS = 231790
USUARIOS_TOTAL_DANE_CNPV2018 = 319306  # validado contra fuente DANE

PERSONAS_POR_HOGAR_MAGDALENA = 3.7  # DANE, CNPV 2018 (promedio departamental)

# Consumo promedio por vivienda (kWh/año)
CONSUMO_PROMEDIO_VIVIENDA_RURAL_kWh = (DEMANDA_RESIDENCIAL_RURAL_GWh * 1_000_000) / USUARIOS_RURALES
CONSUMO_PROMEDIO_VIVIENDA_URBANA_kWh = (DEMANDA_RESIDENCIAL_URBANA_GWh * 1_000_000) / USUARIOS_URBANOS

# ==========================================================
# Funciones del indicador de impacto social
# ==========================================================
def viviendas_rurales_beneficiadas(excedente_electrico_kWh_ano):
    """Número de viviendas rurales equivalentes que podrían abastecerse con el excedente."""
    return excedente_electrico_kWh_ano / CONSUMO_PROMEDIO_VIVIENDA_RURAL_kWh

def personas_rurales_beneficiadas(excedente_electrico_kWh_ano):
    """Número de personas equivalentes beneficiadas (viviendas x personas/hogar DANE)."""
    return viviendas_rurales_beneficiadas(excedente_electrico_kWh_ano) * PERSONAS_POR_HOGAR_MAGDALENA

def porcentaje_cobertura_rural_departamental(excedente_electrico_kWh_ano):
    """% de la demanda residencial rural departamental que cubre el excedente (indicador complementario)."""
    return (excedente_electrico_kWh_ano / (DEMANDA_RESIDENCIAL_RURAL_GWh * 1_000_000)) * 100

# ==========================================================
# Tabla resultado por planta y escenario (se llena con los
# excedentes eléctricos reales que calcule el modelo Pyomo)
# ==========================================================
def construir_tabla_impacto_social(excedentes_dict, nombre_escenario):
    """
    excedentes_dict: {'Planta A': excedente_kWh_ano, 'Planta B': ..., ...}
    nombre_escenario: 'Linea_Base', 'Escenario_1_Termico', o 'Escenario_2_Biogas'
    """
    registros = []
    for planta, excedente in excedentes_dict.items():
        registros.append({
            'Escenario': nombre_escenario,
            'Planta': planta,
            'Excedente_kWh_ano': excedente,
            'Viviendas_rurales_beneficiadas': viviendas_rurales_beneficiadas(excedente),
            'Personas_rurales_beneficiadas': personas_rurales_beneficiadas(excedente),
            'Pct_cobertura_rural_departamental': porcentaje_cobertura_rural_departamental(excedente)
        })
    return pd.DataFrame(registros)

print(f"Consumo promedio vivienda rural: {CONSUMO_PROMEDIO_VIVIENDA_RURAL_kWh:.1f} kWh/año")
print(f"Personas por hogar (Magdalena, DANE CNPV 2018): {PERSONAS_POR_HOGAR_MAGDALENA}")

Consumo promedio vivienda rural: 346.2 kWh/año
Personas por hogar (Magdalena, DANE CNPV 2018): 3.7


In [24]:
import json
datos_sociales = {
    'demanda_rural_GWh': DEMANDA_RESIDENCIAL_RURAL_GWh,
    'demanda_urbana_GWh': DEMANDA_RESIDENCIAL_URBANA_GWh,
    'usuarios_rurales': USUARIOS_RURALES,
    'usuarios_urbanos': USUARIOS_URBANOS,
    'personas_por_hogar': PERSONAS_POR_HOGAR_MAGDALENA
}
with open('datos_sociales.json', 'w') as f:
    json.dump(datos_sociales, f, indent=2)

In [25]:
CONSUMO_ESPECIFICO_ESTERILIZACION_CONTINUA = 315  # kg vapor / tRFF procesado

def vapor_esterilizacion_continua(capacidad_planta_tRFF_h):
    """Vapor requerido para esterilización continua (kg/h), escalado por capacidad de planta."""
    return CONSUMO_ESPECIFICO_ESTERILIZACION_CONTINUA * capacidad_planta_tRFF_h

capacidades = {'Planta A': 30, 'Planta B': 40, 'Planta C': 30, 'Planta D': 24, 'Planta E': 30, 'Planta F': 20}
for planta, cap in capacidades.items():
    vapor = vapor_esterilizacion_continua(cap)
    print(f"{planta}: {vapor:,.0f} kg vapor/h (esterilización continua)")

Planta A: 9,450 kg vapor/h (esterilización continua)
Planta B: 12,600 kg vapor/h (esterilización continua)
Planta C: 9,450 kg vapor/h (esterilización continua)
Planta D: 7,560 kg vapor/h (esterilización continua)
Planta E: 9,450 kg vapor/h (esterilización continua)
Planta F: 6,300 kg vapor/h (esterilización continua)


In [26]:
raw_efi = pd.read_excel('Escenarios.xlsx', sheet_name='Eficiencia', header=None)

plantas = ['Planta A', 'Planta B', 'Planta C', 'Planta D', 'Planta E', 'Planta F']
# Cada planta ocupa 2 columnas (Eléctrica, Térmica), empezando en la columna N (índice 13)
col_inicio = 13

registros_demanda = []
for r in range(5, 23):  # filas 6 a 23 (índice 5 a 22) - de 'Recepción' a 'Total'
    etapa = raw_efi.iloc[r, 12]  # columna M
    if pd.isna(etapa):
        continue
    for i, planta in enumerate(plantas):
        col_e = col_inicio + i*2       # columna eléctrica
        col_t = col_inicio + i*2 + 1   # columna térmica
        valor_e = raw_efi.iloc[r, col_e]
        valor_t = raw_efi.iloc[r, col_t]
        if pd.notna(valor_e):
            registros_demanda.append({'Etapa': etapa, 'Planta': planta, 'Tipo_energia': 'Electrica', 'Valor_kW': valor_e})
        if pd.notna(valor_t):
            registros_demanda.append({'Etapa': etapa, 'Planta': planta, 'Tipo_energia': 'Termica', 'Valor_kW': valor_t})

df_demanda_por_etapa = pd.DataFrame(registros_demanda)

# Separar el resumen (Total, Eficiencia global) de los datos por etapa individual
resumen = df_demanda_por_etapa[df_demanda_por_etapa['Etapa'].isin(['Total', 'Eficiencia global planta extractora'])]
df_demanda_por_etapa = df_demanda_por_etapa[~df_demanda_por_etapa['Etapa'].isin(['Total', 'Eficiencia global planta extractora'])].reset_index(drop=True)

print(df_demanda_por_etapa.shape)
df_demanda_por_etapa.head(20)

(103, 4)


,Etapa,Planta,Tipo_energia,Valor_kW
0,Recepción,Planta A,Electrica,1.000000
1,Recepción,Planta B,Electrica,0.170000
2,Recepción,Planta D,Electrica,0.504101
3,Recepción,Planta E,Electrica,0.200000
4,Recepción,Planta F,Electrica,0.433371
5,Esterilización,Planta A,Electrica,0.080000
6,Esterilización,Planta A,Termica,4866.736111
7,Esterilización,Planta B,Electrica,0.120000
8,Esterilización,Planta B,Termica,6156.944444
9,Esterilización,Planta C,Electrica,0.410000


In [27]:
df_demanda_por_etapa.to_csv('demanda_energia_por_etapa.csv', index=False)

In [28]:
import pandas as pd
import numpy as np

# ==========================================================
# PASO 1: Cargar datos mensuales (RFF evolution, 2022)
# ==========================================================
raw_rff = pd.read_excel('Escenarios.xlsx', sheet_name='RFF evolution', header=None)

meses = raw_rff.iloc[1:13, 1].tolist()  # filas 7-18 (índice 1-12), columna B
plantas = ['Planta A', 'Planta B', 'Planta C', 'Planta D', 'Planta E', 'Planta F']

registros_rff = []
for i, mes in enumerate(meses):
    fila = 1 + i  # índice de fila correspondiente
    for j, planta in enumerate(plantas):
        col = 2 + j  # columnas C-H (índice 2-7)
        valor = raw_rff.iloc[fila, col]
        registros_rff.append({'Mes': mes, 'Mes_num': i+1, 'Planta': planta, 'RFF_ton': valor})

df_rff_mensual = pd.DataFrame(registros_rff)

# ==========================================================
# PASO 2: Factor de estacionalidad (relativo al promedio mensual)
# ==========================================================
promedio_mensual = df_rff_mensual.groupby('Planta')['RFF_ton'].transform('mean')
df_rff_mensual['Factor_estacionalidad'] = df_rff_mensual['RFF_ton'] / promedio_mensual
df_rff_mensual['Excedente_deficit_ton'] = df_rff_mensual['RFF_ton'] - promedio_mensual

print(df_rff_mensual.pivot(index='Mes', columns='Planta', values='Factor_estacionalidad'))

# ==========================================================
# PASO 3: Necesidad de almacenamiento (buffer para generación constante)
# ==========================================================
def biomasa_almacenamiento_requerido(df_planta):
    """
    Calcula el máximo déficit acumulado a lo largo del año -
    equivale a la capacidad de almacenamiento (buffer) necesaria
    para sostener un flujo constante de biomasa/energía todo el año.
    """
    df_planta = df_planta.sort_values('Mes_num')
    acumulado = df_planta['Excedente_deficit_ton'].cumsum()
    buffer_requerido = acumulado.max() - acumulado.min()
    return buffer_requerido

resultados_buffer = []
for planta in plantas:
    df_p = df_rff_mensual[df_rff_mensual['Planta'] == planta]
    buffer_ton = biomasa_almacenamiento_requerido(df_p)
    total_anual = df_p['RFF_ton'].sum()
    resultados_buffer.append({
        'Planta': planta,
        'Buffer_almacenamiento_ton': buffer_ton,
        'Pct_del_total_anual': (buffer_ton / total_anual) * 100
    })

df_buffer = pd.DataFrame(resultados_buffer)
print(df_buffer)

TypeError: agg function failed [how->mean,dtype->object]

In [29]:
import pandas as pd
import numpy as np

# ==========================================================
# PASO 1: Cargar datos mensuales (RFF evolution, 2022)
# ==========================================================
raw_rff = pd.read_excel('Escenarios.xlsx', sheet_name='RFF evolution', header=None)

# CORRECCIÓN: la tabla empieza en la fila 6 de Excel (índice 5 en pandas)
# fila 6 = encabezado, fila 7 (índice 6) = ENERO ... fila 18 (índice 17) = DICIEMBRE
meses = raw_rff.iloc[6:18, 1].tolist()
plantas = ['Planta A', 'Planta B', 'Planta C', 'Planta D', 'Planta E', 'Planta F']

registros_rff = []
for i, mes in enumerate(meses):
    fila = 6 + i  # índice correcto de fila
    for j, planta in enumerate(plantas):
        col = 2 + j  # columnas C-H (índice 2-7)
        valor = raw_rff.iloc[fila, col]
        registros_rff.append({'Mes': mes, 'Mes_num': i+1, 'Planta': planta, 'RFF_ton': valor})

df_rff_mensual = pd.DataFrame(registros_rff)
df_rff_mensual['RFF_ton'] = pd.to_numeric(df_rff_mensual['RFF_ton'], errors='coerce')  # asegurar tipo numérico

print(df_rff_mensual.shape)  # debería ser (72, 4) -> 12 meses x 6 plantas
print(df_rff_mensual.head(15))

# ==========================================================
# PASO 2: Factor de estacionalidad (relativo al promedio mensual)
# ==========================================================
promedio_mensual = df_rff_mensual.groupby('Planta')['RFF_ton'].transform('mean')
df_rff_mensual['Factor_estacionalidad'] = df_rff_mensual['RFF_ton'] / promedio_mensual
df_rff_mensual['Excedente_deficit_ton'] = df_rff_mensual['RFF_ton'] - promedio_mensual

print(df_rff_mensual.pivot(index='Mes', columns='Planta', values='Factor_estacionalidad'))

# ==========================================================
# PASO 3: Necesidad de almacenamiento (buffer para generación constante)
# ==========================================================
def biomasa_almacenamiento_requerido(df_planta):
    df_planta = df_planta.sort_values('Mes_num')
    acumulado = df_planta['Excedente_deficit_ton'].cumsum()
    buffer_requerido = acumulado.max() - acumulado.min()
    return buffer_requerido

resultados_buffer = []
for planta in plantas:
    df_p = df_rff_mensual[df_rff_mensual['Planta'] == planta]
    buffer_ton = biomasa_almacenamiento_requerido(df_p)
    total_anual = df_p['RFF_ton'].sum()
    resultados_buffer.append({
        'Planta': planta,
        'Buffer_almacenamiento_ton': buffer_ton,
        'Pct_del_total_anual': (buffer_ton / total_anual) * 100
    })

df_buffer = pd.DataFrame(resultados_buffer)
print(df_buffer)

(72, 4)
        Mes  Mes_num    Planta    RFF_ton
0     ENERO        1  Planta A   7051.000
1     ENERO        1  Planta B  25334.340
2     ENERO        1  Planta C  14053.000
3     ENERO        1  Planta D  11129.000
4     ENERO        1  Planta E   6475.319
5     ENERO        1  Planta F   3834.000
6   FEBRERO        2  Planta A   7403.000
7   FEBRERO        2  Planta B  21591.000
8   FEBRERO        2  Planta C  14614.000
9   FEBRERO        2  Planta D  11764.000
10  FEBRERO        2  Planta E   7139.000
11  FEBRERO        2  Planta F   3506.550
12    MARZO        3  Planta A  18240.923
13    MARZO        3  Planta B  21943.110
14    MARZO        3  Planta C  17161.000
Planta      Planta A  Planta B  Planta C  Planta D  Planta E  Planta F
Mes                                                                   
ABRIL       1.284151  1.198443  1.091081  1.025341  1.111447  1.094356
AGOSTO      0.923904  0.680771  0.976339  0.859863  0.895418  0.819680
DICIEMBRE   1.115671  0.600619  0.90

In [30]:
df_rff_mensual.to_csv('rff_mensual_estacionalidad.csv', index=False)
df_buffer.to_csv('buffer_almacenamiento_biomasa.csv', index=False)

In [31]:
# Limpiar espacios invisibles (non-breaking space) y forzar tipo numérico
df_tidy['Valor'] = df_tidy['Valor'].apply(lambda v: str(v).replace('\xa0', '').strip() if isinstance(v, str) else v)
df_tidy['Valor'] = pd.to_numeric(df_tidy['Valor'], errors='coerce')

In [32]:
import pandas as pd
import numpy as np

# ==========================================================
# Limpieza completa de Datos_Base (con corrección de texto/nbsp)
# ==========================================================
raw = pd.read_excel('Escenarios.xlsx', sheet_name='Datos_Base', header=None, usecols='A:H')
plantas = raw.iloc[1, 2:8].tolist()

df_datos = raw.iloc[2:].copy()
df_datos.columns = ['Parametro', 'Unidad'] + plantas
df_datos = df_datos.reset_index(drop=True)

# Identificar encabezados de etapa (sin unidad y sin ningún valor en las 6 plantas)
is_header = df_datos['Parametro'].notna() & df_datos['Unidad'].isna() & df_datos[plantas].isna().all(axis=1)
df_datos['Etapa'] = df_datos['Parametro'].where(is_header).ffill()
df_datos = df_datos[~is_header].reset_index(drop=True)

# Rellenar nombre de parámetro para las filas "huérfanas" (valor específico por tRFF)
df_datos['Parametro'] = df_datos['Parametro'].ffill()
dup_mask = df_datos.duplicated(subset=['Etapa', 'Parametro'], keep='first')
df_datos.loc[dup_mask, 'Parametro'] = df_datos.loc[dup_mask, 'Parametro'] + ' (específico por tRFF)'
df_datos.loc[dup_mask, 'Unidad'] = df_datos.loc[dup_mask, 'Unidad'].fillna('por tRFF procesado')

# Formato tidy
df_tidy = df_datos.melt(id_vars=['Parametro', 'Unidad', 'Etapa'], var_name='Planta', value_name='Valor')
df_tidy['Etapa'] = df_tidy['Etapa'].fillna('General')

# --- CORRECCIÓN: limpiar espacios invisibles (\xa0) y forzar tipo numérico ---
df_tidy['Valor'] = df_tidy['Valor'].apply(lambda v: str(v).replace('\xa0', '').strip() if isinstance(v, str) else v)
df_tidy['Valor'] = pd.to_numeric(df_tidy['Valor'], errors='coerce')

# Separar filas con dato realmente faltante (para no perderlas silenciosamente)
faltantes = df_tidy[df_tidy['Valor'].isna() & (df_tidy['Etapa'] != 'General')]
df_tidy = df_tidy.dropna(subset=['Valor']).reset_index(drop=True)

print("Filas con dato faltante en el Excel original:")
print(faltantes[['Parametro', 'Unidad', 'Etapa']].drop_duplicates())

# Corregir unidad de "Humedad de secado" (error de tecleo en el Excel original: °C -> %)
df_tidy.loc[df_tidy['Parametro'] == 'Humedad de secado', 'Unidad'] = '%'

# Agregar de vuelta "Flujo de aire de secado" como dato pendiente explícito (NaN documentado)
fila_pendiente = pd.DataFrame({
    'Parametro': ['Flujo de aire de secado'] * len(plantas),
    'Unidad': ['kgaire seco∙h-1'] * len(plantas),
    'Etapa': ['Secado de nueces'] * len(plantas),
    'Planta': plantas,
    'Valor': [np.nan] * len(plantas)
})

# Separar eficiencias reportadas (no se usan como dato de entrada - se recalculan desde cero)
es_eficiencia = df_tidy['Parametro'].str.contains('Eficiencia', case=False, na=False)
df_datos_base = df_tidy[~es_eficiencia].reset_index(drop=True)
df_datos_base = pd.concat([df_datos_base, fila_pendiente], ignore_index=True)

print("\ndf_datos_base listo:", df_datos_base.shape)

# ==========================================================
# Verificación específica: Presión/Temperatura/Flujo de vapor de Planta A
# ==========================================================
verificacion = df_datos_base[
    (df_datos_base['Parametro'].isin(['Presión de vapor', 'Temperatura de vapor', 
                                        'Temperatura agua de alimentación', 'Flujo de vapor'])) &
    (df_datos_base['Planta'] == 'Planta A')
]
print("\nVerificación Planta A (deben ser números, no texto):")
print(verificacion)

Filas con dato faltante en el Excel original:
                                            Parametro              Unidad  \
15       Eficiencia energpetica (específico por tRFF)  por tRFF procesado   
41                            Flujo de aire de secado     kgaire seco∙h-1   
55  Temperatura agua de alimentación (específico p...  por tRFF procesado   
76                                        Humedad MPD                   %   

               Etapa  
15    Esterilización  
41  Secado de nueces  
55           Caldera  
76        Desfrutado  

df_datos_base listo: (313, 5)

Verificación Planta A (deben ser números, no texto):
                           Parametro       Unidad    Etapa    Planta    Valor
48                    Flujo de vapor  kgvapor∙h-1  Caldera  Planta A  71984.0
49                  Presión de vapor          bar  Caldera  Planta A      3.0
50              Temperatura de vapor           °C  Caldera  Planta A    300.0
51  Temperatura agua de alimentación           °C  Calde

In [33]:
from iapws import IAPWS97

vapor = IAPWS97(P=24*0.1, T=256+273.15)   # 24 bar, 256°C
agua = IAPWS97(P=1*0.1, T=95+273.15)      # agua de alimentación, 95°C

print(f"h_vapor (IAPWS97): {vapor.h:.1f} kJ/kg   | Excel reporta: 2897.3")
print(f"h_agua (IAPWS97): {agua.h:.1f} kJ/kg   | Excel reporta: 398")

ModuleNotFoundError: No module named 'iapws'

In [1]:
from iapws import IAPWS97

vapor = IAPWS97(P=24*0.1, T=256+273.15)
agua = IAPWS97(P=1*0.1, T=95+273.15)

print(f"h_vapor (IAPWS97): {vapor.h:.1f} kJ/kg   | Excel reporta: 2897.3")
print(f"h_agua (IAPWS97): {agua.h:.1f} kJ/kg   | Excel reporta: 398")

h_vapor (IAPWS97): 2901.7 kJ/kg   | Excel reporta: 2897.3
h_agua (IAPWS97): 398.0 kJ/kg   | Excel reporta: 398


In [3]:
import pandas as pd
from iapws import IAPWS97

def entalpia_vapor(P_bar, T_C):
    return IAPWS97(P=P_bar*0.1, T=T_C+273.15).h

from iapws import IAPWS97

def entalpia_vapor(P_bar, T_C):
    return IAPWS97(P=P_bar*0.1, T=T_C+273.15).h

def entalpia_agua_alimentacion(T_C, P_bar=1.0):
    return IAPWS97(P=P_bar*0.1, T=T_C+273.15).h

def eficiencia_caldera_ASME(m_vapor_kg_h, P_bar, T_vapor_C, T_agua_C, energia_combustible_kWth):
    h_f = entalpia_vapor(P_bar, T_vapor_C)
    h_i = entalpia_agua_alimentacion(T_agua_C)
    energia_salida_kWth = m_vapor_kg_h * (h_f - h_i) / 3600  # kJ/h -> kW
    eta = energia_salida_kWth / energia_combustible_kWth
    return eta, h_f, h_i

# Datos línea base (hoja Eficiencia, filas 151-169) - las 6 plantas
datos_caldera_LB = {
    'Planta A': {'m_vapor': 13842,    'P_bar': 24,   'T_vapor': 256,   'T_agua': 95, 'energia_comb': 12691.45},
    'Planta B': {'m_vapor': 19056,    'P_bar': 11,   'T_vapor': 188,   'T_agua': 92, 'energia_comb': 22014.43},
    'Planta C': {'m_vapor': 14127,    'P_bar': 9.6,  'T_vapor': 182.4, 'T_agua': 80, 'energia_comb': 14251.88},
    'Planta D': {'m_vapor': 9878.4,   'P_bar': 9.2,  'T_vapor': 180,   'T_agua': 85, 'energia_comb': 12278.74},
    'Planta E': {'m_vapor': 15480,    'P_bar': 11.3, 'T_vapor': 189.1, 'T_agua': 90, 'energia_comb': 18579.14},
    'Planta F': {'m_vapor': 9720,     'P_bar': 10.3, 'T_vapor': 181.2, 'T_agua': 80, 'energia_comb': 9813.09},
}

resultados_eficiencia = []
for planta, d in datos_caldera_LB.items():
    eta, h_f, h_i = eficiencia_caldera_ASME(d['m_vapor'], d['P_bar'], d['T_vapor'], d['T_agua'], d['energia_comb'])
    resultados_eficiencia.append({'Planta': planta, 'Eficiencia_termica_pct': eta*100, 'h_f_kJkg': h_f, 'h_i_kJkg': h_i})

df_eficiencia_caldera_LB = pd.DataFrame(resultados_eficiencia)
print(df_eficiencia_caldera_LB)

     Planta  Eficiencia_termica_pct     h_f_kJkg    h_i_kJkg
0  Planta A               75.851854  2901.723409  398.030275
1  Planta B               57.850513  2791.347175  385.403703
2  Planta C               67.511090  2786.874036  334.990544
3  Planta D               54.252441  2783.645373  355.979148
4  Planta E               55.896269  2792.118503  376.991515
5  Planta F               67.226038  2778.305957  334.990544
